# EDA on Retail Sales Data
OIBSIP · Data Analytics · Level 1 · Task 1

**Dataset:** *Retail Sales Dataset* from Kaggle, saved as `retail_sales_dataset.csv` next to this notebook. Expected columns: `Transaction ID, Date, Customer ID, Gender, Age, Product Category, Quantity, Price per Unit, Total Amount`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

## 1. Load and inspect

In [ ]:
df = pd.read_csv("retail_sales_dataset.csv")
df["Date"] = pd.to_datetime(df["Date"])
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nNull values:\n", df.isnull().sum())
df.head()

## 2. Descriptive statistics

In [ ]:
num_cols = df.select_dtypes(include="number").columns.drop(["Transaction ID", "Customer ID"], errors="ignore")
stats = df[num_cols].agg(["mean", "median", "std"]).T
stats["mode"] = [df[c].mode().iloc[0] for c in num_cols]
stats.round(2)

## 3. Time series: monthly and quarterly sales

In [ ]:
monthly = df.set_index("Date")["Total Amount"].resample("MS").sum()
quarterly = df.set_index("Date")["Total Amount"].resample("QS").sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
monthly.plot(ax=axes[0], marker="o", title="Monthly sales")
quarterly.plot(ax=axes[1], marker="o", title="Quarterly sales")
for ax in axes: ax.set_ylabel("Total amount")
plt.tight_layout(); plt.show()

**Observation:** write which months/quarters peak and any seasonality you notice.

## 4. Customer demographics

In [ ]:
bins = [17, 25, 35, 45, 55, 100]
labels = ["18-25", "26-35", "36-45", "46-55", "56+"]
df["Age Group"] = pd.cut(df["Age"], bins=bins, labels=labels)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(df["Age"], bins=15, ax=axes[0]); axes[0].set_title("Age distribution")
df["Age Group"].value_counts().sort_index().plot(kind="bar", ax=axes[1], title="Customers by age group")
df["Gender"].value_counts().plot(kind="pie", autopct="%1.0f%%", ax=axes[2], title="Gender split")
axes[2].set_ylabel("")
plt.tight_layout(); plt.show()

## 5. Product analysis

In [ ]:
by_cat = df.groupby("Product Category")["Total Amount"].sum().sort_values(ascending=False)
by_cat.plot(kind="bar", title="Revenue by product category", ylabel="Total amount")
plt.show()

# The dataset has few categories; show up to the top 10 by units sold.
top = df.groupby("Product Category")["Quantity"].sum().sort_values(ascending=False).head(10)
print("Top product categories by units sold:\n", top)

## 6. Correlation heatmap

In [ ]:
corr = df[["Age", "Quantity", "Price per Unit", "Total Amount"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation matrix"); plt.show()

## 7. Extra insight: average spend by age group and gender

In [ ]:
pivot = df.pivot_table(index="Age Group", columns="Gender", values="Total Amount", aggfunc="mean", observed=True)
pivot.plot(kind="bar", title="Average transaction value by age group and gender", ylabel="Average amount")
plt.show()

## 8. Conclusions and recommendations
Replace these with findings from your own charts:

1. **Stock for peak periods.** Increase inventory ahead of the highest-revenue months.
2. **Target the highest-spending age group** with loyalty offers or bundles.
3. **Promote the top revenue category** and review under-performing categories for discounts or removal.